# 03 -- Layer 1 Pipeline + AWIK Score Ranking

Runs `run_awik_pipeline` (K-Means -> DBSCAN -> Mahalanobis, empirical threshold,
`src/awik/layer1.py`) across all 75 weekly windows (ISO week-year indexing -- see `features.py`), then builds the triage ranking.

In [1]:
import sys
sys.path.insert(0, '../src')
import time
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from awik.config import DEFAULT_MAH_THRESH
from awik.features import ALL_FEATURES
from awik.layer1 import run_awik_pipeline
from awik import io

df_master = io.load_df('01_df_master')
weeks_df = io.load_df('01_weeks_df')
N_MASTER = len(df_master)

In [2]:
print(f"-> [Layer 1] AWIK Two-Stage Spatial Clustering ({len(weeks_df)} weeks)...")
start = time.time()
df_semua, df_threats = run_awik_pipeline(df_master, ALL_FEATURES, weeks_df)
print(f"=> Done in {time.time()-start:.2f}s")
print(f"   Rows processed : {len(df_semua):,}")
print(f"   Total flagged  : {len(df_threats):,}")
print(f"   Flagging rate  : {len(df_threats)/len(df_semua)*100:.2f}%")
print(f"   Mahalanobis threshold: {DEFAULT_MAH_THRESH} (empirical, label-free)")

-> [Layer 1] AWIK Two-Stage Spatial Clustering (75 weeks)...


=> Done in 25.20s
   Rows processed : 283,519
   Total flagged  : 10,161
   Flagging rate  : 3.58%
   Mahalanobis threshold: 37.44 (empirical, label-free)


## AWIK Score ranking (triage output, not the flagging decision itself)

In [3]:
df_suspects = df_threats.groupby(['User_ID','Role','Department']).agg(
    Max_Mahalanobis   = ('Mahalanobis_Sq', 'max'),
    Anomaly_Frequency = ('Year_Week', 'count'),
    Openness          = ('Openness', 'mean'),
    Conscientiousness = ('Conscientiousness', 'mean'),
    Agreeableness     = ('Agreeableness', 'mean'),
    Neuroticism       = ('Neuroticism', 'mean'),
).reset_index()

df_suspects['Psychometric_Risk'] = (
    df_suspects['Neuroticism'] - df_suspects['Agreeableness'] - df_suspects['Conscientiousness'])
scaler_awik = MinMaxScaler(feature_range=(0, 100))
df_suspects[['Spatial_Score','Psych_Score']] = scaler_awik.fit_transform(
    df_suspects[['Max_Mahalanobis','Psychometric_Risk']])

# AWIK_Score only orders the triage queue -- the 0.70/0.30 weighting does NOT
# determine l1_flagged membership; that comes from AWIK_Flag (Mahalanobis/DBSCAN) above.
df_suspects['AWIK_Score'] = df_suspects['Spatial_Score'] * 0.70 + df_suspects['Psych_Score'] * 0.30
df_ranking = df_suspects.sort_values('AWIK_Score', ascending=False).reset_index(drop=True)

l1_flagged = set(df_threats['User_ID'].unique())

def flagged_set_for_weight(w_spatial):
    s = df_suspects['Spatial_Score'] * w_spatial + df_suspects['Psych_Score'] * (1 - w_spatial)
    return set(df_suspects.loc[s.notna(), 'User_ID'])

inv = all(flagged_set_for_weight(w) == l1_flagged for w in [0.5, 0.7, 1.0])
print(f"Total users flagged at Layer 1: {len(l1_flagged):,}")
print(f"Flagged set invariant to weight in {{0.5,0.7,1.0}}: {inv}  (weight only orders triage)")
df_ranking[['User_ID','Role','Anomaly_Frequency','Spatial_Score','Psych_Score','AWIK_Score']].head(15)

Total users flagged at Layer 1: 1,056
Flagged set invariant to weight in {0.5,0.7,1.0}: True  (weight only orders triage)


,User_ID,Role,Anomaly_Frequency,Spatial_Score,Psych_Score,AWIK_Score
0,ORM1484,StockroomClerk,40,100.000000,40.00,82.000000
1,UXC1495,StockroomClerk,44,63.931388,20.00,50.751971
2,TWM2199,ProductionLineWorker,49,50.524019,37.50,46.616813
3,BSM2844,ProductionLineWorker,51,43.324891,47.50,44.577424
4,GDM3960,StockroomClerk,38,26.409843,77.50,41.736890
5,ETG2821,ProductionLineWorker,36,19.822773,91.25,41.250941
6,SKS0339,StockroomClerk,28,20.543426,80.00,38.380399
7,HHF1995,ComputerScientist,21,20.855325,68.75,35.223728
8,AIJ3995,ProductionLineWorker,44,15.842059,77.50,34.339441
9,ADP3667,StockroomClerk,22,10.015939,90.00,34.011157


In [4]:
io.save_df(df_semua, '03_df_semua')
io.save_df(df_threats, '03_df_threats')
io.save_df(df_suspects, '03_df_suspects')
io.save_df(df_ranking, '03_df_ranking')
io.save_json({'l1_flagged': sorted(l1_flagged), 'N_MASTER': N_MASTER}, '03_layer1_state')
print("Checkpoint saved -> data/interim/03_*")

Checkpoint saved -> data/interim/03_*
